以下是任务一相关代码。

In [1]:
import polars as pl
import glob
import os

# 获取所有 parquet 文件路径
parquet_files = glob.glob(r"G:\Users\caoruijie\big data\clean_data_partitioned\**\*.parquet", recursive=True)

# 构建所有 LazyFrame，并补充分区字段
lazy_frames = []
for file in parquet_files:
    # 提取分区字段
    parts = file.split(os.sep)
    behavior_type = None
    for part in parts:
        if part.startswith("behavior_type="):
            behavior_type = part.split("=")[1]
            break
    lf = pl.scan_parquet(file).with_columns(pl.lit(behavior_type).alias("behavior_type"))
    lazy_frames.append(lf)

# 合并所有 LazyFrame
df = pl.concat(lazy_frames)

# 去重前行数
count_before = df.select(pl.len()).collect()[0, 0]

# 精确去重
df_unique = df.unique(subset=["user_id", "item_id", "behavior_type", "timestamp"])

# 去重后行数
count_after = df_unique.select(pl.len()).collect()[0, 0]

print(f"去重前行数: {count_before}")
print(f"去重后行数: {count_after}")

去重前行数: 100150489
去重后行数: 100150440


以下是任务二相关代码。

In [5]:
import polars as pl
import glob
import os

# 获取所有 parquet 文件路径
parquet_files = glob.glob(r"G:\Users\caoruijie\big data\clean_data_partitioned\**\*.parquet", recursive=True)

# 构建所有 LazyFrame，并补充分区字段
lazy_frames = []
for file in parquet_files:
    parts = file.split(os.sep)
    behavior_type = None
    for part in parts:
        if part.startswith("behavior_type="):
            behavior_type = part.split("=")[1]
            break
    lf = pl.scan_parquet(file).with_columns(pl.lit(behavior_type).alias("behavior_type"))
    lazy_frames.append(lf)

# 合并所有 LazyFrame
df = pl.concat(lazy_frames)

# 计算时间差和 session_id
df = (
    df.sort(["user_id", "timestamp"])
    .with_columns([
        (pl.col("timestamp") - pl.col("timestamp").shift(1).over("user_id")).alias("timediff"),
    ])
    .with_columns([
        ((pl.col("timediff").is_null()) | (pl.col("timediff") > 1800)).cast(pl.Int32).alias("is_new_session")
    ])
    .with_columns([
        pl.cum_sum("is_new_session").alias("session_id")
    ])
)

# 查看前几行
print(df.head(10).collect())

shape: (10, 8)
┌─────────┬─────────┬─────────────┬────────────┬─────────────┬──────────┬─────────────┬────────────┐
│ user_id ┆ item_id ┆ merchant_id ┆ timestamp  ┆ behavior_ty ┆ timediff ┆ is_new_sess ┆ session_id │
│ ---     ┆ ---     ┆ ---         ┆ ---        ┆ pe          ┆ ---      ┆ ion         ┆ ---        │
│ i32     ┆ i32     ┆ i32         ┆ i32        ┆ ---         ┆ i32      ┆ ---         ┆ i32        │
│         ┆         ┆             ┆            ┆ str         ┆          ┆ i32         ┆            │
╞═════════╪═════════╪═════════════╪════════════╪═════════════╪══════════╪═════════════╪════════════╡
│ 1       ┆ 2268318 ┆ 2520377     ┆ 1511544070 ┆ pv          ┆ null     ┆ 1           ┆ 1          │
│ 1       ┆ 2333346 ┆ 2520771     ┆ 1511561733 ┆ pv          ┆ 17663    ┆ 1           ┆ 2          │
│ 1       ┆ 2576651 ┆ 149192      ┆ 1511572885 ┆ pv          ┆ 11152    ┆ 1           ┆ 3          │
│ 1       ┆ 3830808 ┆ 4181361     ┆ 1511593493 ┆ pv          ┆ 20608    ┆ 1 

以下是任务三相关代码

In [13]:
import polars as pl
import glob
import os

# 获取所有 parquet 文件路径
parquet_files = glob.glob(r"G:\Users\caoruijie\big data\clean_data_partitioned\**\*.parquet", recursive=True)

# 构建所有 LazyFrame，并补充分区字段
lazy_frames = []
for file in parquet_files:
    parts = file.split(os.sep)
    behavior_type = None
    for part in parts:
        if part.startswith("behavior_type="):
            behavior_type = part.split("=")[1]
            break
    lf = pl.scan_parquet(file).with_columns(pl.lit(behavior_type).alias("behavior_type"))
    lazy_frames.append(lf)

# 合并所有 LazyFrame
df_lazy = pl.concat(lazy_frames)

# 只保留 user_id 和行为类型
df = df_lazy.select(["user_id", "behavior_type"]).collect()
print(type(df))  # 检查类型
print(df.__class__.__module__)  # 检查类型来源

# 计算每个用户的所有行为集合（确保 df 是 polars.DataFrame）
user_behaviors = df.group_by("user_id").agg([
    pl.col("behavior_type").unique().alias("behaviors")
])

# 1. 有多少独立用户进行了 PV？
pv_users = user_behaviors.filter(user_behaviors["behaviors"].list.contains("pv"))
pv_user_count = pv_users.height

# 2. 这些 PV 用户中，有多少人进一步产生了 Fav 或 Cart 行为？
pv_fav_cart_users = pv_users.filter(
    (pv_users["behaviors"].list.contains("fav")) | (pv_users["behaviors"].list.contains("cart"))
)
pv_fav_cart_user_count = pv_fav_cart_users.height

# 3. 最终有多少人完成了 Buy？
pv_fav_cart_buy_users = pv_fav_cart_users.filter(
    pv_fav_cart_users["behaviors"].list.contains("buy")
)
pv_fav_cart_buy_user_count = pv_fav_cart_buy_users.height

# 阶段转化率
rate_fav_cart = pv_fav_cart_user_count / pv_user_count if pv_user_count else 0
rate_buy = pv_fav_cart_buy_user_count / pv_fav_cart_user_count if pv_fav_cart_user_count else 0

print(f"独立PV用户数: {pv_user_count}")
print(f"PV用户中产生Fav或Cart的用户数: {pv_fav_cart_user_count}，转化率: {rate_fav_cart:.2%}")
print(f"最终完成Buy的用户数: {pv_fav_cart_buy_user_count}，转化率: {rate_buy:.2%}")

<class 'polars.dataframe.frame.DataFrame'>
polars.dataframe.frame
独立PV用户数: 984114
PV用户中产生Fav或Cart的用户数: 855502，转化率: 86.93%
最终完成Buy的用户数: 600279，转化率: 70.17%


以下是任务四相关代码

在电商日志数据中，常见暗示爬虫/机器人账号的特征有：

单账号短时间内请求量极高（如 PV 数远超正常用户）。
行为单一（如只浏览不下单、不加购、不收藏）。
行为时间分布异常（如深夜大量活跃）。
访问商品/页面的种类极多或极少。
user_agent、IP、设备等信息异常（如同一IP下多个账号）。
在没有 user_agent、IP 等字段时，最常用的维度是 user_id 的请求量（如 PV 数），通常“高频PV账号”最可疑。

Polars 筛选高频PV账号代码如下（假设只用 user_id 和行为类型）：

In [1]:
import polars as pl
import glob
import os

# 获取所有 parquet 文件路径
parquet_files = glob.glob(r"G:\Users\caoruijie\big data\clean_data_partitioned\**\*.parquet", recursive=True)
lazy_frames = []
for file in parquet_files:
    parts = file.split(os.sep)
    behavior_type = None
    for part in parts:
        if part.startswith("behavior_type="):
            behavior_type = part.split("=")[1]
            break
    lf = pl.scan_parquet(file).with_columns(pl.lit(behavior_type).alias("behavior_type"))
    lazy_frames.append(lf)

df_lazy = pl.concat(lazy_frames)
df = df_lazy.select(["user_id", "behavior_type"]).collect()

# 统计每个 user_id 的 PV 请求数
pv_counts = (
    df.filter(pl.col("behavior_type") == "pv")
      .group_by("user_id")
      .count()
      .rename({"count": "pv_count"})
)

# 设定高频点击阈值（如PV数大于500为嫌疑账号）
suspect_threshold = 500
suspects = pv_counts.filter(pl.col("pv_count") > suspect_threshold)

# 统计嫌疑账号贡献的总PV数及占比
total_pv = pv_counts["pv_count"].sum()
suspect_pv = suspects["pv_count"].sum()
suspect_ratio = suspect_pv / total_pv if total_pv else 0

print(f"嫌疑账号数: {suspects.height}")
print(f"嫌疑账号贡献PV数: {suspect_pv}")
print(f"嫌疑账号贡献PV占比: {suspect_ratio:.2%}")

C:\Users\caoruijie\AppData\Local\Temp\ipykernel_560\1718879868.py:25: DeprecationWarning: `GroupBy.count` was renamed; use `GroupBy.len` instead
  .count()


嫌疑账号数: 1738
嫌疑账号贡献PV数: 979731
嫌疑账号贡献PV占比: 1.09%


任务五

In [1]:
import time
start = time.time()
import polars as pl
import os
import duckdb

# ===================== 路径配置（和你原始完全一致） =====================
INPUT_ROOT = r"G:\Users\caoruijie\big data\clean_data_partitioned"
OUTPUT_ROOT = r"G:\Users\caoruijie\big data\m1_final_clean"
behavior_types = ["pv", "cart", "fav", "buy"]

# ===================== 1. 找出嫌疑用户（低内存） =====================
print("正在识别异常刷PV用户...")

df = pl.scan_parquet(
    os.path.join(INPUT_ROOT, "**/*.parquet"),
    hive_partitioning=True
)

# 只找出 PV>500 的异常用户 ID
suspect_users = (
    df.filter(pl.col("behavior_type") == "pv")
      .group_by("user_id")
      .agg(pl.len().alias("pv_count"))
      .filter(pl.col("pv_count") > 500)
      .select("user_id")
)

# ===================== 2. 清洗全数据（完全保持原始存储模式） =====================
print("正在清洗并输出最终数据...")

for bt in behavior_types:
    # 读取原始分区文件
    lf = pl.scan_parquet(os.path.join(INPUT_ROOT, f"behavior_type={bt}", "*.parquet"))

    # 核心清洗（和你原来逻辑完全一样）
    lf_clean = (
        lf.unique(subset=["user_id", "item_id", "timestamp"])  # 去重
        .filter(pl.col("timestamp") > 0)                       # 剔除无效时间
        .join(suspect_users, on="user_id", how="anti")          # 删掉异常用户
    )

    # ==============================================
    # 关键：完全保持你原始的 Parquet 写入模式
    # 不排序、不修改压缩、不改变任何格式
    # ==============================================
    out_dir = os.path.join(OUTPUT_ROOT, f"behavior_type={bt}")
    os.makedirs(out_dir, exist_ok=True)
    
    # 完全和你原来一样的写入方式，无任何优化
    lf_clean.sink_parquet(
        os.path.join(out_dir, "data.parquet"),
    )

# ===================== 3. 统计结果 =====================
total_rows = 0
for bt in behavior_types:
    f = os.path.join(OUTPUT_ROOT, f"behavior_type={bt}", "data.parquet")
    total_rows += pl.scan_parquet(f).select(pl.len()).collect()[0, 0]

total_size = sum(os.path.getsize(os.path.join(r, f)) for r, _, fs in os.walk(OUTPUT_ROOT) for f in fs)
total_size_gb = total_size / 1024**3

# ===================== 最终输出 =====================
print("-" * 50)
print(f"最终数据总量(Row Count)：{total_rows:,}")
print(f"最终存储体积：{total_size_gb:.3f} GB")
print("-" * 50)
print(f"任务五耗时: {time.time() - start:.2f} 秒")

正在识别异常刷PV用户...
正在清洗并输出最终数据...
--------------------------------------------------
最终数据总量(Row Count)：99,122,540
最终存储体积：0.964 GB
--------------------------------------------------
任务五耗时: 20.86 秒
